# LLM Finetuning Sprint

## 1. Installing Dependencies

In [ ]:
%%capture

# Core training libraries
!pip install -q \
    transformers==4.44.2 \
    datasets==2.20.0 \
    tokenizers==0.19.1 \
    accelerate==0.34.2 \
    peft==0.12.0 \
    trl==0.9.6 \
    bitsandbytes==0.46.1 \
    evaluate==0.4.2

# Utilities
!pip install -q \
    "numpy<2.0.0"\
    pandas \
    scikit-learn \
    rich \
    pyyaml \
    python-dotenv \
    tqdm

# Evaluation
!pip install -q pydantic>=2.9 openai rouge-score

print("✅ Installation complete!")
print("✅ All dependencies compatible (pydantic v2 + openai)")

## 2. Environment & GPU Check

In [ ]:
import sys
import torch

print("="*60)
print("ENVIRONMENT CHECK")
print("="*60)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ WARNING: CUDA not available. Training will be VERY slow on CPU.")

print("="*60)

## 3. Seeds & Determinism

In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42

# Set environment variable for Python hash seed
os.environ['PYTHONHASHSEED'] = str(SEED)

# Set seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # Note: These settings may impact performance
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"✅ Seeds set to {SEED} for reproducibility")
print("⚠️ Note: Full determinism on GPU is not guaranteed due to non-deterministic operations")

## 4. Hugging Face Login

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!hf auth login --token $HF_TOKEN

print("ℹ️ Hugging Face login skipped. Uncomment login() to push models to Hub.")

## 5. Configuration

In [ ]:
from pathlib import Path
from pprint import pprint

# Auto-detect compute dtype (BF16 requires compute capability >= 8.0)
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "artifacts").exists() and (PROJECT_ROOT.parent / "artifacts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

UBER_DATASET_PATH = PROJECT_ROOT / "artifacts" / "data_factory" / "uber_dataset.json"
TRAIN_JSONL_PATH = PROJECT_ROOT / "artifacts" / "train" / "train.jsonl"
GOLDEN_TEST_SET_PATH = PROJECT_ROOT / "artifacts" / "test" / "golden_test_set.jsonl"

CONFIG = {
    # Model
    "base_model": "meta-llama/Meta-Llama-3-8B-Instruct",

    # Local Uber data
    "train_files": [str(UBER_DATASET_PATH), str(TRAIN_JSONL_PATH)],
    "eval_file": str(GOLDEN_TEST_SET_PATH),

    # Tokenization
    "max_length": 512,

    # Training
    "num_train_epochs": 1,
    "max_steps": 120,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "save_steps": 50,
    "eval_steps": 50,
    "save_total_limit": 2,

    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],

    # Quantization
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": compute_dtype,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,

    # Output
    "output_dir": "outputs/adapter",
    "push_to_hub": False,

    # Generation
    "max_new_tokens": 128,
    "temperature": 0.0,
    "do_sample": False,
}

# Effective batch size
effective_batch_size = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]

print("="*60)
print("CONFIGURATION")
print("="*60)
pprint(CONFIG)
print("="*60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Uber dataset: {UBER_DATASET_PATH}")
print(f"Train JSONL: {TRAIN_JSONL_PATH}")
print(f"Golden test set: {GOLDEN_TEST_SET_PATH}")
print(f"Compute dtype: {compute_dtype}")
print(f"Using BF16: {use_bf16}")
print(f"Effective batch size: {effective_batch_size}")
print("="*60)